# HyperSHAP: Example for Optuna Integration

In this example, we demonstrate how to load data from an [optuna](https://optuna.org/) study directly into HyperSHAP for downstream hyperparameter analysis.

This is useful when you have already run an optuna HPO study and want to understand *why* certain hyperparameters matter more than others — without having to redefine a `ConfigSpace` manually.

> **Prerequisites:** `optuna` must be installed.
> ```bash
> pip install optuna
> # or
> pip install hypershap[optuna]
> ```

## Step 1 — Run an optuna study

We first set up a small synthetic objective and run an optuna study to mimic a realistic HPO scenario.
The objective uses a float, an integer, and a categorical hyperparameter — the same types supported by the optuna integration.

In [ ]:
from __future__ import annotations

import math

import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress per-trial logs


def objective(trial: optuna.Trial) -> float:
    """Synthetic objective that mimics a tunable ML algorithm.

    Hyperparameters
    ---------------
    a : float  in [0.1, 1.5]   — learning-rate-like continuous parameter
    b : int    in [2, 10]      — depth-like integer parameter
    c : str    in {"X", "Y"}   — algorithm variant (categorical)
    """
    a = trial.suggest_float("a", 0.1, 1.5)
    b = trial.suggest_int("b", 2, 10)
    c = trial.suggest_categorical("c", ["X", "Y"])

    # Variant X: performance mainly driven by b, slightly by a
    if c == "X":
        return math.sin(a) + b
    # Variant Y: interaction between a and b dominates
    return math.cos(a * b) + 1.5


# Run a maximisation study with 200 trials
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=200)

print(f"Completed trials : {len(study.trials)}")
print(f"Best value       : {study.best_value:.4f}")
print(f"Best params      : {study.best_params}")

## Step 2 — Load the study into HyperSHAP

`from_optuna_study` is the main entry point for the optuna integration. It:

1. Extracts a `ConfigurationSpace` from the trial distributions.
2. Converts all completed trial results into `(Configuration, float)` pairs.
3. Fits a surrogate model (default: `RandomForestRegressor`) on those pairs.
4. Returns an `ExplanationTask` ready for HyperSHAP analysis.

For **minimisation** studies (`direction="minimize"`) the objective values are automatically negated so that HyperSHAP's *higher-is-better* convention is respected. Pass `negate=False` to disable this.

In [ ]:
from hypershap import HyperSHAP, from_optuna_study

# One-liner: study → ExplanationTask
explanation_task = from_optuna_study(study)

print("Config space HPs :", explanation_task.get_hyperparameter_names())
print("Number of HPs    :", explanation_task.get_num_hyperparameters())

hypershap = HyperSHAP(explanation_task=explanation_task)

## Step 3 — Tunability analysis

For tunability we need a **baseline configuration** — the starting point from which we measure how much tuning each hyperparameter can improve performance. A natural choice is the *default configuration* of the inferred `ConfigurationSpace` (i.e. the midpoint/default of each hyperparameter range), which represents the algorithm before any tuning has taken place.

In [ ]:
# Use the default configuration of the inferred ConfigSpace as the baseline —
# this represents the algorithm with no tuning applied.
default_config = explanation_task.config_space.get_default_configuration()
print("Default (baseline) config:", default_config)

iv_tunability = hypershap.tunability(baseline_config=default_config)
print(iv_tunability)

### Visualisations

In [ ]:
hypershap.plot_si_graph()

In [ ]:
hypershap.plot_stacked_bar()

## Step 4 — Ablation analysis

We can also run an ablation analysis to understand which hyperparameters are responsible for the performance gain from the default configuration to the best configuration found by optuna.

In [ ]:
from ConfigSpace import Configuration

best_config = Configuration(
    explanation_task.config_space,
    values=study.best_params,
)

iv_ablation = hypershap.ablation(
    config_of_interest=best_config,  # optimized config found by optuna
    baseline_config=default_config,  # default / untuned starting point
)
print(iv_ablation)

In [ ]:
hypershap.plot_waterfall()

## Step 5 — Advanced: using the lower-level helpers

If you need more control — e.g. to inspect the inferred `ConfigurationSpace`, filter trials manually, or pass a custom surrogate model — you can use the lower-level helpers directly.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

from hypershap.optuna_task import study_to_config_space, study_to_data

# 1. Inspect the inferred configuration space
cs = study_to_config_space(study)
print("Inferred ConfigurationSpace:")
print(cs)

# 2. Convert trials to (Configuration, float) pairs — apply custom filtering if needed
data = study_to_data(study, config_space=cs)
print(f"\nConverted {len(data)} trials to (Configuration, float) pairs.")

# 3. Build an ExplanationTask with a custom surrogate model
from hypershap.task import ExplanationTask

custom_task = ExplanationTask.from_data(
    config_space=cs,
    data=data,
    base_model=GradientBoostingRegressor(n_estimators=200, random_state=0),
)

hs_custom = HyperSHAP(explanation_task=custom_task)
iv_custom = hs_custom.tunability(baseline_config=default_config)
print("\nTunability with GradientBoostingRegressor surrogate:")
print(iv_custom)